# Revue du dataset

Description des fichiers presents dans `data/` : apercu (head) et statistiques de base pour chaque fichier parquet.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATA_DIR = Path("data")

df_patient = pd.read_parquet(DATA_DIR / "patient.parquet")
df_sejour = pd.read_parquet(DATA_DIR / "sejour.parquet")
df_mouvement = pd.read_parquet(DATA_DIR / "mouvement.parquet")
df_document = pd.read_parquet(DATA_DIR / "document.parquet")
df_biologie = pd.read_parquet(DATA_DIR / "biologie.parquet")
df_medicament = pd.read_parquet(DATA_DIR / "medicament.parquet")
df_constante = pd.read_parquet(DATA_DIR / "constante.parquet")

fichiers = {
    "patient": df_patient,
    "sejour": df_sejour,
    "mouvement": df_mouvement,
    "document": df_document,
    "biologie": df_biologie,
    "medicament": df_medicament,
    "constante": df_constante,
}

## Vue d'ensemble

Nombre de lignes / colonnes par fichier.

In [2]:
resume = pd.DataFrame({
    "fichier": list(fichiers.keys()),
    "n_lignes": [df.shape[0] for df in fichiers.values()],
    "n_colonnes": [df.shape[1] for df in fichiers.values()],
    "colonnes": [list(df.columns) for df in fichiers.values()],
})
resume

,fichier,n_lignes,n_colonnes,colonnes
0,patient,100,4,"[id_patient, age, sexe, date_deces]"
1,sejour,100,7,"[id_sejour, id_patient, date_entree, date_sort..."
2,mouvement,100,5,"[id_mouvement, id_sejour, uf, date_entree, dat..."
3,document,158,6,"[id_entrepot, id_patient, id_sejour, titre, ty..."
4,biologie,488,8,"[id_biologie, id_patient, id_sejour, uf, date_..."
5,medicament,329,11,"[id_medicament, id_patient, id_sejour, date_ad..."
6,constante,2684,7,"[id_constante, id_patient, id_sejour, type_con..."


## 1. `patient.parquet`

Une ligne par patient : age, sexe, date de deces eventuelle.

In [3]:
df_patient.head()

,id_patient,age,sexe,date_deces
0,PAT000001,62,M,NaT
1,PAT000002,57,F,NaT
2,PAT000003,37,F,NaT
3,PAT000004,66,F,NaT
4,PAT000005,16,M,NaT


In [4]:
df_patient.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id_patient  100 non-null    object        
 1   age         100 non-null    int64         
 2   sexe        100 non-null    object        
 3   date_deces  1 non-null      datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 3.2+ KB


In [5]:
df_patient.describe(include="all")

,id_patient,age,sexe,date_deces
count,100,100.000000,100,1
unique,100,NaN,2,NaN
top,PAT000001,NaN,M,NaN
freq,1,NaN,53,NaN
mean,NaN,54.550000,NaN,2024-04-06 08:36:14
min,NaN,5.000000,NaN,2024-04-06 08:36:14
25%,NaN,39.000000,NaN,2024-04-06 08:36:14
50%,NaN,56.000000,NaN,2024-04-06 08:36:14
75%,NaN,71.500000,NaN,2024-04-06 08:36:14
max,NaN,85.000000,NaN,2024-04-06 08:36:14


In [6]:
print("Valeurs manquantes par colonne :")
df_patient.isna().sum()

Valeurs manquantes par colonne :


id_patient     0
age            0
sexe           0
date_deces    99
dtype: int64

In [7]:
print("Repartition du sexe :")
print(df_patient["sexe"].value_counts())
print()
print(f"Part de patients decedes : {df_patient['date_deces'].notna().mean():.1%}")

Repartition du sexe :
sexe
M    53
F    47
Name: count, dtype: int64

Part de patients decedes : 1.0%


## 2. `sejour.parquet`

Une ligne par sejour hospitalier : dates d'entree/sortie, UF d'entree/sortie.

In [8]:
df_sejour.head()

,id_sejour,id_patient,date_entree,date_sortie,uf_entree,uf_sortie,specialite
0,SEJ000001,PAT000001,2023-12-25 15:08:34,2023-12-31 15:08:34,1710,1710,MEDECINE INTERNE
1,SEJ000002,PAT000002,2023-08-13 12:53:17,2023-08-15 12:53:17,1130,1130,CANCERO ADULTE
2,SEJ000003,PAT000003,2024-09-04 22:15:21,2024-09-11 22:15:21,1850,1850,OBSTETRIQUE
3,SEJ000004,PAT000004,2024-07-20 02:38:59,2024-07-29 02:38:59,1570,1570,HEPATO-GASTRO-ENTERO
4,SEJ000005,PAT000005,2024-06-16 10:19:09,2024-06-19 10:19:09,1940,1940,PNEUMOLOGIE


In [9]:
df_sejour.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id_sejour    100 non-null    object        
 1   id_patient   100 non-null    object        
 2   date_entree  100 non-null    datetime64[ns]
 3   date_sortie  100 non-null    datetime64[ns]
 4   uf_entree    100 non-null    object        
 5   uf_sortie    100 non-null    object        
 6   specialite   100 non-null    object        
dtypes: datetime64[ns](2), object(5)
memory usage: 5.6+ KB


In [10]:
duree_sejour = (df_sejour["date_sortie"] - df_sejour["date_entree"]).dt.days
print("Duree de sejour (jours) :")
duree_sejour.describe()

Duree de sejour (jours) :


count    100.000000
mean       8.770000
std        8.527857
min        1.000000
25%        3.000000
50%        6.000000
75%       10.500000
max       36.000000
dtype: float64

In [11]:
print(f"Nombre de sejours par patient (moyenne) : {df_sejour.groupby('id_patient').size().mean():.2f}")
print()
print("Sejours par UF d'entree :")
df_sejour["uf_entree"].value_counts()

Nombre de sejours par patient (moyenne) : 1.00

Sejours par UF d'entree :


uf_entree
1130    14
1350    10
1940     9
1440     9
1240     9
1520     8
1710     6
1850     5
1570     5
1800     4
1610     4
1640     3
1990     3
1760     3
1960     2
1730     2
1150     2
1650     1
1250     1
Name: count, dtype: int64

## 3. `mouvement.parquet`

Une ligne par mouvement intra-sejour (changement d'UF).

In [12]:
df_mouvement.head()

,id_mouvement,id_sejour,uf,date_entree,date_sortie
0,MVT000001,SEJ000001,1710,2023-12-25 15:08:34,2023-12-31 15:08:34
1,MVT000002,SEJ000002,1130,2023-08-13 12:53:17,2023-08-15 12:53:17
2,MVT000003,SEJ000003,1850,2024-09-04 22:15:21,2024-09-11 22:15:21
3,MVT000004,SEJ000004,1570,2024-07-20 02:38:59,2024-07-29 02:38:59
4,MVT000005,SEJ000005,1940,2024-06-16 10:19:09,2024-06-19 10:19:09


In [13]:
df_mouvement.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id_mouvement  100 non-null    object        
 1   id_sejour     100 non-null    object        
 2   uf            100 non-null    object        
 3   date_entree   100 non-null    datetime64[ns]
 4   date_sortie   100 non-null    datetime64[ns]
dtypes: datetime64[ns](2), object(3)
memory usage: 4.0+ KB


In [14]:
print(f"Nombre de mouvements par sejour (moyenne) : {df_mouvement.groupby('id_sejour').size().mean():.2f}")
print()
print("Mouvements par UF :")
df_mouvement["uf"].value_counts()

Nombre de mouvements par sejour (moyenne) : 1.00

Mouvements par UF :


uf
1130    14
1350    10
1940     9
1440     9
1240     9
1520     8
1710     6
1850     5
1570     5
1800     4
1610     4
1640     3
1990     3
1760     3
1960     2
1730     2
1150     2
1650     1
1250     1
Name: count, dtype: int64

## 4. `document.parquet`

Une ligne par document (compte-rendu, courrier, etc.) rattache a un sejour.

In [15]:
df_document.drop(columns=["texte_affichage"]).head()

,id_entrepot,id_patient,id_sejour,titre,type_document
0,DOC000001,PAT000001,SEJ000001,Compte-rendu d'hospitalisation - 2023-12-31,Compte-rendu d'hospitalisation
1,DOC000002,PAT000002,SEJ000002,Compte-rendu d'hospitalisation - 2023-08-15,Compte-rendu d'hospitalisation
2,DOC000003,PAT000003,SEJ000003,Compte-rendu d'hospitalisation - 2024-09-11,Compte-rendu d'hospitalisation
3,DOC000004,PAT000004,SEJ000004,Compte-rendu d'hospitalisation - 2024-07-29,Compte-rendu d'hospitalisation
4,DOC000005,PAT000005,SEJ000005,Compte-rendu d'hospitalisation - 2024-06-19,Compte-rendu d'hospitalisation


In [16]:
df_document.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_entrepot      158 non-null    object
 1   id_patient       158 non-null    object
 2   id_sejour        158 non-null    object
 3   titre            158 non-null    object
 4   type_document    158 non-null    object
 5   texte_affichage  158 non-null    object
dtypes: object(6)
memory usage: 7.5+ KB


In [17]:
print("Repartition par type de document :")
print(df_document["type_document"].value_counts())
print()
longueur_texte = df_document["texte_affichage"].str.len()
print("Longueur du texte HTML (caracteres) :")
longueur_texte.describe()

Repartition par type de document :
type_document
Compte-rendu d'hospitalisation    100
Compte-rendu de consultation       29
Compte-rendu operatoire            29
Name: count, dtype: int64

Longueur du texte HTML (caracteres) :


count      158.000000
mean      2903.797468
std       2163.151693
min        368.000000
25%       1316.000000
50%       2466.500000
75%       3796.250000
max      13108.000000
Name: texte_affichage, dtype: float64

## Exemple de contenu d'un document

In [18]:
from IPython.display import HTML
HTML(df_document.iloc[0]["texte_affichage"])

## 5. `biologie.parquet`

Une ligne par resultat de biologie rattache a un sejour : valeur numerique ou textuelle selon le type de resultat.

In [19]:
df_biologie.head()

,id_biologie,id_patient,id_sejour,uf,date_prelevement,valeur_numerique,valeur_texte,unite
0,BIO000001,PAT000001,SEJ000001,1710,2023-12-30 03:08:20,55.08,None,mmol/L
1,BIO000002,PAT000001,SEJ000001,1710,2023-12-26 06:04:10,135.37,None,%
2,BIO000003,PAT000001,SEJ000001,1710,2023-12-28 04:35:22,6.45,None,g/L
3,BIO000004,PAT000001,SEJ000001,1710,2023-12-28 16:44:25,120.44,None,%
4,BIO000005,PAT000001,SEJ000001,1710,2023-12-29 13:47:20,140.29,None,UI/L


In [20]:
df_biologie.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 488 entries, 0 to 487
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_biologie       488 non-null    object        
 1   id_patient        488 non-null    object        
 2   id_sejour         488 non-null    object        
 3   uf                488 non-null    object        
 4   date_prelevement  488 non-null    datetime64[ns]
 5   valeur_numerique  367 non-null    float64       
 6   valeur_texte      121 non-null    object        
 7   unite             367 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 30.6+ KB


In [21]:
print(f"Nombre de resultats de biologie par sejour (moyenne) : {df_biologie.groupby('id_sejour').size().mean():.2f}")
print()
n_numerique = df_biologie["valeur_numerique"].notna().sum()
n_texte = df_biologie["valeur_texte"].notna().sum()
print(f"Resultats numeriques : {n_numerique} ({n_numerique / len(df_biologie):.1%})")
print(f"Resultats textuels   : {n_texte} ({n_texte / len(df_biologie):.1%})")
print()
print("Repartition des valeurs textuelles :")
print(df_biologie["valeur_texte"].value_counts())
print()
print("Repartition des unites (resultats numeriques) :")
print(df_biologie["unite"].value_counts())

Nombre de resultats de biologie par sejour (moyenne) : 4.88

Resultats numeriques : 367 (75.2%)
Resultats textuels   : 121 (24.8%)

Repartition des valeurs textuelles :
valeur_texte
normal             30
positif            24
negatif            23
anormal            23
non contributif    21
Name: count, dtype: int64

Repartition des unites (resultats numeriques) :
unite
g/L       61
10^9/L    59
UI/L      54
umol/L    54
%         53
mmol/L    45
mg/L      41
Name: count, dtype: int64


## 6. `medicament.parquet`

Une ligne par administration medicamenteuse rattachee a un sejour (code UCD/ATC, quantite, unite).

In [22]:
df_medicament.head()

,id_medicament,id_patient,id_sejour,date_administration,quantite_administree,unite,ucd,atc,voie_administration,conditionnelle,commentaire
0,MED000001,PAT000001,SEJ000001,2023-12-26 13:47:10,130,mg,3400921959219,B01AC06,intramusculaire,True,None
1,MED000002,PAT000001,SEJ000001,2023-12-30 18:34:05,16,mg,3400937246430,C09AA02,topique,False,traitement interrompu a la demande du patient
2,MED000003,PAT000001,SEJ000001,2023-12-26 09:19:18,58,mg,3400930096770,C10AA05,orale,False,None
3,MED000004,PAT000002,SEJ000002,2023-08-14 22:58:13,13,mg,3400927699023,N02AA01,topique,False,dose ajustee sur avis medical
4,MED000005,PAT000002,SEJ000002,2023-08-14 08:19:03,53,mg,3400930096770,C10AA05,orale,False,None


In [23]:
df_medicament.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 329 entries, 0 to 328
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_medicament         329 non-null    object        
 1   id_patient            329 non-null    object        
 2   id_sejour             329 non-null    object        
 3   date_administration   329 non-null    datetime64[ns]
 4   quantite_administree  329 non-null    int64         
 5   unite                 329 non-null    object        
 6   ucd                   329 non-null    object        
 7   atc                   329 non-null    object        
 8   voie_administration   329 non-null    object        
 9   conditionnelle        329 non-null    bool          
 10  commentaire           41 non-null     object        
dtypes: bool(1), datetime64[ns](1), int64(1), object(8)
memory usage: 26.1+ KB


In [24]:
print(f"Nombre d'administrations par sejour (moyenne) : {df_medicament.groupby('id_sejour').size().mean():.2f}")
print()
print("Repartition par code ATC :")
print(df_medicament["atc"].value_counts())
print()
print("Repartition par voie d'administration :")
print(df_medicament["voie_administration"].value_counts())
print()
n_conditionnelle = df_medicament["conditionnelle"].sum()
print(f"Administrations conditionnelles (si besoin) : {n_conditionnelle} ({n_conditionnelle / len(df_medicament):.1%})")
print(f"Administrations avec commentaire : {df_medicament['commentaire'].notna().sum()}")

Nombre d'administrations par sejour (moyenne) : 3.29

Repartition par code ATC :
atc
N05BA01    42
N02AA01    37
A02BC01    35
C10AA05    35
M01AE01    34
B01AC06    33
C09AA02    32
J01CA04    30
A10BA02    27
N02BE01    24
Name: count, dtype: int64

Repartition par voie d'administration :
voie_administration
orale              76
intramusculaire    74
sous-cutanee       62
topique            59
intraveineuse      58
Name: count, dtype: int64

Administrations conditionnelles (si besoin) : 53 (16.1%)
Administrations avec commentaire : 41


## 7. `constante.parquet`

Une ligne par constante vitale mesuree pendant le sejour (poids, taille, temperature, frequence cardiaque, frequence respiratoire, saturation en oxygene, pression arterielle), au format long.

In [25]:
df_constante.head()

,id_constante,id_patient,id_sejour,type_constante,date,valeur,unite
0,CST000001,PAT000001,SEJ000001,poids,2023-12-31 00:05:45,101.0,kg
1,CST000002,PAT000001,SEJ000001,poids,2023-12-29 03:13:32,57.5,kg
2,CST000003,PAT000001,SEJ000001,poids,2023-12-26 01:16:20,48.0,kg
3,CST000004,PAT000001,SEJ000001,taille,2023-12-30 07:43:44,163.0,cm
4,CST000005,PAT000001,SEJ000001,temperature,2023-12-30 19:42:26,36.4,degC


In [26]:
df_constante.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2684 entries, 0 to 2683
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_constante    2684 non-null   object        
 1   id_patient      2684 non-null   object        
 2   id_sejour       2684 non-null   object        
 3   type_constante  2684 non-null   object        
 4   date            2684 non-null   datetime64[ns]
 5   valeur          2684 non-null   float64       
 6   unite           2684 non-null   object        
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 146.9+ KB


In [27]:
print("Nombre de mesures par type de constante :")
print(df_constante["type_constante"].value_counts())
print()
print("Statistiques de la valeur par type de constante :")
df_constante.groupby(["type_constante", "unite"])["valeur"].describe()

Nombre de mesures par type de constante :
type_constante
saturation_o2                      414
frequence_cardiaque                411
pression_arterielle_diastolique    408
pression_arterielle_systolique     396
temperature                        377
frequence_respiratoire             375
poids                              203
taille                             100
Name: count, dtype: int64

Statistiques de la valeur par type de constante :


,,count,mean,std,min,25%,50%,75%,max
type_constante,unite,,,,,,,,
frequence_cardiaque,bpm,411.0,84.708029,20.187956,50.0,67.00,85.0,102.00,120.0
frequence_respiratoire,/min,375.0,19.917333,4.854139,12.0,16.00,20.0,24.00,28.0
poids,kg,203.0,80.333990,18.689394,45.6,65.10,82.1,96.25,109.9
pression_arterielle_diastolique,mmHg,408.0,79.022059,16.613159,50.0,65.00,79.0,92.25,110.0
pression_arterielle_systolique,mmHg,396.0,135.123737,26.236458,90.0,112.75,134.0,159.00,180.0
saturation_o2,%,414.0,94.903382,2.843001,90.0,92.00,95.0,97.00,100.0
taille,cm,100.0,171.330000,12.902944,150.0,160.00,171.0,181.50,194.0
temperature,degC,377.0,37.489920,1.156781,35.5,36.50,37.4,38.50,39.5
